# Notebook 5 — Free Energy Surfaces & Committor Visualization

This notebook produces the key analysis figures:
1. **2D free energy surface** — histogram of trajectory density in feature space,  
   converted to free energy via F = -kT·ln(P) (kT = 0.593 kcal/mol at 300 K).
2. **Committor-colored contours** — free energy contour lines overlaid on a  
   smooth color map of committor probability, revealing the transition state ensemble.

The top features identified by SHAP (Notebook 4) are the natural axes for these plots.

In [ ]:
import yaml
import sys
sys.path.insert(0, "..")
import pandas as pd

from src.contours import plot_free_energy_2d, plot_contours

In [ ]:
with open("../config/chignolin.yaml") as f:
    cfg = yaml.safe_load(f)

SYSTEM = cfg["system"]
N = 40

DIST_CSV         = f"../data/{SYSTEM}/distances_ON.csv"
TORS_CSV         = f"../data/{SYSTEM}/backbone_torsions.csv"
CLUSTER_PROB_CSV = f"../data/{SYSTEM}/cluster_{N}_prob.csv"

df_dist = pd.read_csv(DIST_CSV)
df_tors = pd.read_csv(TORS_CSV)
df_all = pd.concat([df_dist, df_tors], axis=1)
print(f"Feature matrix: {df_all.shape}")

## 2D Free Energy Surface

Choose two features (e.g. the top two from the SHAP plot in Notebook 4).  
For Chignolin the key distances are between backbone O/N atoms across the β-turn.

In [ ]:
import re

# Sanitize column names for easy access
df_all.columns = [re.sub(":", "_", c) for c in df_all.columns]
print(df_all.columns[:10].tolist())  # inspect available features

In [ ]:
# --- Select two features for the 2D plot ---
# Replace these with the top-2 features from your SHAP plot:
FEAT1 = df_all.columns[0]  # e.g. top SHAP distance
FEAT2 = df_all.columns[1]  # e.g. second SHAP distance

plot_free_energy_2d(
    df_all[FEAT1].values,
    df_all[FEAT2].values,
    bins=50,
    xlabel=FEAT1,
    ylabel=FEAT2,
    title=f"Free energy — {SYSTEM}"
)

## Committor-colored Contour Plot

The transition state ensemble (q ≈ 0.5) appears as the white region  
between the blue (unfolded) and red (folded) zones.

In [ ]:
# The cluster_prob_csv must be at the same stride as df_all
plot_contours(
    df_all,
    trait1=FEAT1,
    trait2=FEAT2,
    committor_csv=CLUSTER_PROB_CSV
)

## Top features from SHAP (loop over pairs)

After running Notebook 4, use the SHAP summary to identify the top N features  
and produce all pairwise free energy / committor plots.

In [ ]:
# Paste the top features from your SHAP summary here:
# (Chignolin canonical top-10 distances from Best_Analysis/Free Energy.ipynb)
top_features = [
    "DIST_ THR 8 O 119 - TYR 2 N 23",
    "DIST_ TYR 2 N 23 - TRP 9 N 120",
    "DIST_ ASP 3 O 55 - THR 6 N 85",
    "DIST_ TYR 2 OH 36 - TRP 9 NE1 130",
    "DIST_ TYR 2 OH 36 - TYR 10 N 147",
]
# Sanitize column names to match
top_features = [re.sub(":", "_", f) for f in top_features]

# Plot first feature pair
for f1, f2 in [(top_features[0], top_features[1]),
               (top_features[0], top_features[2])]:
    if f1 in df_all.columns and f2 in df_all.columns:
        plot_contours(df_all, f1, f2, CLUSTER_PROB_CSV)